In [3]:
!nvidia-smi

Wed Aug 26 20:16:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.43.02              KMD Version: 610.43.02     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:01:00.0 Off |                   On |
| N/A   31C    P0             41W /  400W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

In [4]:
!nvcc --version

sh: 1: nvcc: not found


In [5]:
!find /usr -name "libcudart.so*" 2>/dev/null | head -20

In [6]:
!find /usr -name "cuda_runtime.h" 2>/dev/null | head -20

In [7]:
!gcc --version

gcc (conda-forge gcc 15.2.0-7) 15.2.0
Copyright (C) 2025 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [8]:
!ldconfig -p | grep -E 'libcuda|libnvrtc|libcudart'

	libcudadebugger.so.1 (libc6,x86-64) => /usr/lib64/libcudadebugger.so.1
	libcuda.so.1 (libc6,x86-64) => /usr/lib64/libcuda.so.1
	libcuda.so (libc6,x86-64) => /usr/lib64/libcuda.so


In [9]:
!which conda

/srv/conda/condabin/conda


In [10]:
!conda list | grep -Ei 'cuda|cudatoolkit|nvrtc|cupy|numba'

numba                               0.67.0           py311hdfc3925_1          conda-forge
nvidia-cuda-cupti-cu12              12.1.105         pypi_0                   pypi
nvidia-cuda-nvrtc-cu12              12.1.105         pypi_0                   pypi
nvidia-cuda-runtime-cu12            12.1.105         pypi_0                   pypi


In [11]:
!python -c "import cupy; print('CuPy:', cupy.__version__); print('CUDA:', cupy.cuda.runtime.runtimeGetVersion())"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'cupy'


In [12]:
!echo $CONDA_PREFIX

/srv/conda/envs/notebook


In [13]:
!find $CONDA_PREFIX -name "libnvrtc.so*" 2>/dev/null

/srv/conda/envs/notebook/lib/python3.11/site-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12


In [14]:
!find $CONDA_PREFIX -name "libcudart.so*" 2>/dev/null

/srv/conda/envs/notebook/lib/python3.11/site-packages/nvidia/cuda_runtime/lib/libcudart.so.12


In [15]:
!find $CONDA_PREFIX -name "cuda_runtime.h" 2>/dev/null

/srv/conda/envs/notebook/lib/python3.11/site-packages/triton/backends/nvidia/include/cuda_runtime.h
/srv/conda/envs/notebook/lib/python3.11/site-packages/nvidia/cuda_runtime/include/cuda_runtime.h


In [16]:
!which python

/srv/conda/envs/notebook/bin/python


In [17]:
!python --version

Python 3.11.15


In [18]:
!find $CONDA_PREFIX -name "nvrtc.h" -o -name "cuda.h" 2>/dev/null

/srv/conda/envs/notebook/x86_64-conda-linux-gnu/sysroot/usr/include/linux/cuda.h
/srv/conda/envs/notebook/include/hwloc/cuda.h
/srv/conda/envs/notebook/lib/python3.11/site-packages/triton/backends/nvidia/include/cuda.h
/srv/conda/envs/notebook/lib/python3.11/site-packages/torch/include/torch/csrc/api/include/torch/cuda.h
/srv/conda/envs/notebook/lib/python3.11/site-packages/nvidia/cuda_runtime/include/cuda.h
/srv/conda/envs/notebook/lib/python3.11/site-packages/nvidia/cuda_nvrtc/include/nvrtc.h


In [19]:
!find $CONDA_PREFIX -name "libcuda.so*" -o -name "libnvrtc.so*" -o -name "libcudart.so*" 2>/dev/null

/srv/conda/envs/notebook/lib/python3.11/site-packages/nvidia/cuda_runtime/lib/libcudart.so.12
/srv/conda/envs/notebook/lib/python3.11/site-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12


In [20]:
#include <stdio.h>
#include <dlfcn.h>

int main2() {
    void *h = dlopen("libnvrtc.so.12", RTLD_NOW);

    if (!h) {
        printf("FAILED: %s\n", dlerror());
        return 1;
    }

    printf("SUCCESS: NVRTC loaded!\n");

    dlclose(h);
    return 0;
}

In [21]:
main2()

FAILED: libnvrtc.so.12: cannot open shared object file: No such file or directory


In [22]:

int main3() {
    const char *path =
        "/srv/conda/envs/notebook/lib/python3.11/site-packages/"
        "nvidia/cuda_nvrtc/lib/libnvrtc.so.12";

    void *h = dlopen(path, RTLD_NOW);

    if (!h) {
        printf("FAILED: %s\n", dlerror());
        return 1;
    }

    printf("SUCCESS: NVRTC loaded!\n");

    dlclose(h);
    return 0;
}

In [23]:
main3()

SUCCESS: NVRTC loaded!


In [24]:


int main4() {
    void *h = dlopen("libcuda.so.1", RTLD_NOW);

    if (!h) {
        printf("FAILED: %s\n", dlerror());
        return 1;
    }

    printf("SUCCESS: CUDA driver loaded!\n");

    dlclose(h);
    return 0;
}

In [25]:
main4()

SUCCESS: CUDA driver loaded!


In [26]:


typedef int CUresult;
typedef int CUdevice;

#define CUDA_SUCCESS 0

int gpu_test2()
{
    void *cuda = dlopen("libcuda.so.1", RTLD_NOW);

    if (!cuda) {
        printf("FAILED to load CUDA driver: %s\n", dlerror());
        return 1;
    }

    typedef CUresult (*cuInit_t)(unsigned int);
    typedef CUresult (*cuDeviceGetCount_t)(int *);
    typedef CUresult (*cuDeviceGet_t)(CUdevice *, int);
    typedef CUresult (*cuDeviceGetName_t)(char *, int, CUdevice);

    cuInit_t cuInit =
        (cuInit_t)dlsym(cuda, "cuInit");

    cuDeviceGetCount_t cuDeviceGetCount =
        (cuDeviceGetCount_t)dlsym(cuda, "cuDeviceGetCount");

    cuDeviceGet_t cuDeviceGet =
        (cuDeviceGet_t)dlsym(cuda, "cuDeviceGet");

    cuDeviceGetName_t cuDeviceGetName =
        (cuDeviceGetName_t)dlsym(cuda, "cuDeviceGetName");

    if (!cuInit || !cuDeviceGetCount ||
        !cuDeviceGet || !cuDeviceGetName) {

        printf("FAILED to find CUDA driver functions\n");
        dlclose(cuda);
        return 1;
    }

    CUresult r = cuInit(0);

    if (r != CUDA_SUCCESS) {
        printf("cuInit FAILED: %d\n", r);
        dlclose(cuda);
        return 1;
    }

    int count = 0;

    r = cuDeviceGetCount(&count);

    if (r != CUDA_SUCCESS) {
        printf("cuDeviceGetCount FAILED: %d\n", r);
        dlclose(cuda);
        return 1;
    }

    printf("CUDA devices visible: %d\n", count);

    for (int i = 0; i < count; i++) {

        CUdevice dev;
        char name[256];

        cuDeviceGet(&dev, i);
        cuDeviceGetName(name, sizeof(name), dev);

        printf("GPU %d: %s\n", i, name);
    }

    dlclose(cuda);

    return 0;
}

In [27]:
gpu_test2()

CUDA devices visible: 16
GPU 0: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 1: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 2: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 3: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 4: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 5: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 6: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 7: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 8: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 9: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 10: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 11: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 12: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 13: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 14: NVIDIA A100-SXM4-40GB MIG 1g.5gb
GPU 15: NVIDIA A100-SXM4-40GB MIG 1g.5gb


In [28]:


#include <string.h>


typedef int CUresult;
typedef int CUdevice;
typedef void* CUcontext;
typedef void* CUmodule;
typedef void* CUfunction;
typedef unsigned long long CUdeviceptr;

#define CUDA_SUCCESS 0

typedef CUresult (*PFN_cuInit)(unsigned int);
typedef CUresult (*PFN_cuDeviceGetCount)(int *);
typedef CUresult (*PFN_cuDeviceGet)(CUdevice *, int);
typedef CUresult (*PFN_cuDeviceGetName)(char *, int, CUdevice);
typedef CUresult (*PFN_cuCtxCreate)(CUcontext *, unsigned int, CUdevice);
typedef CUresult (*PFN_cuMemAlloc)(CUdeviceptr *, size_t);
typedef CUresult (*PFN_cuMemFree)(CUdeviceptr);
typedef CUresult (*PFN_cuMemcpyHtoD)(CUdeviceptr, const void *, size_t);
typedef CUresult (*PFN_cuMemcpyDtoH)(void *, CUdeviceptr, size_t);
typedef CUresult (*PFN_cuModuleLoadData)(CUmodule *, const void *);
typedef CUresult (*PFN_cuModuleGetFunction)(CUfunction *, CUmodule, const char *);
typedef CUresult (*PFN_cuLaunchKernel)(
    CUfunction,
    unsigned int, unsigned int, unsigned int,
    unsigned int, unsigned int, unsigned int,
    unsigned int,
    void *,
    void **,
    void **
);
typedef CUresult (*PFN_cuCtxSynchronize)(void);
typedef CUresult (*PFN_cuModuleUnload)(CUmodule);
typedef CUresult (*PFN_cuCtxDestroy)(CUcontext);

#define LOAD(name) do { \
    name = (PFN_##name)dlsym(cuda, #name "_v2"); \
    if (!name) name = (PFN_##name)dlsym(cuda, #name); \
    if (!name) { printf("Missing %s\n", #name); return 1; } \
} while(0)

int gpu_test()
{
    void *cuda = dlopen("libcuda.so.1", RTLD_NOW);

    if (!cuda) {
        printf("CUDA driver load failed: %s\n", dlerror());
        return 1;
    }

    PFN_cuInit cuInit;
    PFN_cuDeviceGetCount cuDeviceGetCount;
    PFN_cuDeviceGet cuDeviceGet;
    PFN_cuDeviceGetName cuDeviceGetName;
    PFN_cuCtxCreate cuCtxCreate;
    PFN_cuMemAlloc cuMemAlloc;
    PFN_cuMemFree cuMemFree;
    PFN_cuMemcpyHtoD cuMemcpyHtoD;
    PFN_cuMemcpyDtoH cuMemcpyDtoH;
    PFN_cuModuleLoadData cuModuleLoadData;
    PFN_cuModuleGetFunction cuModuleGetFunction;
    PFN_cuLaunchKernel cuLaunchKernel;
    PFN_cuCtxSynchronize cuCtxSynchronize;
    PFN_cuModuleUnload cuModuleUnload;
    PFN_cuCtxDestroy cuCtxDestroy;

    LOAD(cuInit);
    LOAD(cuDeviceGetCount);
    LOAD(cuDeviceGet);
    LOAD(cuDeviceGetName);
    LOAD(cuCtxCreate);
    LOAD(cuMemAlloc);
    LOAD(cuMemFree);
    LOAD(cuMemcpyHtoD);
    LOAD(cuMemcpyDtoH);
    LOAD(cuModuleLoadData);
    LOAD(cuModuleGetFunction);
    LOAD(cuLaunchKernel);
    LOAD(cuCtxSynchronize);
    LOAD(cuModuleUnload);
    LOAD(cuCtxDestroy);

    if (cuInit(0) != CUDA_SUCCESS) {
        printf("cuInit failed\n");
        return 1;
    }

    int count = 0;
    cuDeviceGetCount(&count);

    printf("CUDA devices: %d\n", count);

    CUdevice dev;
    cuDeviceGet(&dev, 0);

    char name[256];
    cuDeviceGetName(name, sizeof(name), dev);

    printf("Using: %s\n", name);

    CUcontext ctx;
    cuCtxCreate(&ctx, 0, dev);

    /*
       Tiny GPU kernel.
       Each GPU thread performs one addition.
    */
    const char *src =
        "extern \"C\" __global__ "
        "void add(float *a, float *b, float *c) {"
        "int i = blockIdx.x * blockDim.x + threadIdx.x;"
        "c[i] = a[i] + b[i];"
        "}";

    /*
       Load NVRTC WITHOUT cuda.h or nvrtc.h.
    */
    void *nvrtc = dlopen(
        "/srv/conda/envs/notebook/lib/python3.11/site-packages/"
        "nvidia/cuda_nvrtc/lib/libnvrtc.so.12",
        RTLD_NOW
    );

    if (!nvrtc) {
        printf("NVRTC load failed: %s\n", dlerror());
        return 1;
    }

    typedef int (*CreateProgram)(
        void **, const char *, const char *,
        int, const char **, const char **
    );

    typedef int (*CompileProgram)(
        void *, int, const char **
    );

    typedef int (*GetPTXSize)(
        void *, size_t *
    );

    typedef int (*GetPTX)(
        void *, char *
    );

    typedef int (*GetLogSize)(
        void *, size_t *
    );

    typedef int (*GetLog)(
        void *, char *
    );

    typedef int (*DestroyProgram)(
        void **
    );

    CreateProgram create =
        (CreateProgram)dlsym(nvrtc, "nvrtcCreateProgram");

    CompileProgram compile =
        (CompileProgram)dlsym(nvrtc, "nvrtcCompileProgram");

    GetPTXSize get_ptx_size =
        (GetPTXSize)dlsym(nvrtc, "nvrtcGetPTXSize");

    GetPTX get_ptx =
        (GetPTX)dlsym(nvrtc, "nvrtcGetPTX");

    GetLogSize get_log_size =
        (GetLogSize)dlsym(nvrtc, "nvrtcGetProgramLogSize");

    GetLog get_log =
        (GetLog)dlsym(nvrtc, "nvrtcGetProgramLog");

    DestroyProgram destroy =
        (DestroyProgram)dlsym(nvrtc, "nvrtcDestroyProgram");

    void *program;

    int r = create(
        &program,
        src,
        "add.cu",
        0,
        NULL,
        NULL
    );

    if (r != 0) {
        printf("NVRTC create failed: %d\n", r);
        return 1;
    }

    const char *opts[] = {
        "--gpu-architecture=compute_80"
    };

    r = compile(program, 1, opts);

    if (r != 0) {

        size_t log_size = 0;
        get_log_size(program, &log_size);

        char *log = malloc(log_size + 1);
        get_log(program, log);
        log[log_size] = '\0';

        printf("NVRTC compile failed:\n%s\n", log);

        free(log);
        return 1;
    }

    size_t ptx_size;
    get_ptx_size(program, &ptx_size);

    char *ptx = malloc(ptx_size);
    get_ptx(program, ptx);

    CUmodule module;
    CUfunction kernel;

    if (cuModuleLoadData(&module, ptx) != CUDA_SUCCESS) {
        printf("PTX load failed\n");
        return 1;
    }

    if (cuModuleGetFunction(
            &kernel,
            module,
            "add") != CUDA_SUCCESS) {

        printf("Kernel lookup failed\n");
        return 1;
    }

    /*
       Very small test first.
    */
    const int N = 1024;

    float *a = malloc(N * sizeof(float));
    float *b = malloc(N * sizeof(float));
    float *c = malloc(N * sizeof(float));

    for (int i = 0; i < N; i++) {
        a[i] = i;
        b[i] = 2.0f * i;
    }

    CUdeviceptr da, db, dc;

    cuMemAlloc(&da, N * sizeof(float));
    cuMemAlloc(&db, N * sizeof(float));
    cuMemAlloc(&dc, N * sizeof(float));

    cuMemcpyHtoD(da, a, N * sizeof(float));
    cuMemcpyHtoD(db, b, N * sizeof(float));

    void *args[] = {
        &da,
        &db,
        &dc
    };

    r = cuLaunchKernel(
        kernel,
        4, 1, 1,
        256, 1, 1,
        0,
        NULL,
        args,
        NULL
    );

    if (r != CUDA_SUCCESS) {
        printf("Kernel launch failed: %d\n", r);
        return 1;
    }

    cuCtxSynchronize();

    cuMemcpyDtoH(
        c,
        dc,
        N * sizeof(float)
    );

    int errors = 0;

    for (int i = 0; i < N; i++) {
        float expected = a[i] + b[i];

        if (c[i] != expected) {
            errors++;
        }
    }

    printf("\n============================\n");
    printf("GPU COMPUTATION TEST\n");
    printf("============================\n");
    printf("N          : %d\n", N);
    printf("GPU        : %s\n", name);
    printf("Verification: %s\n",
           errors == 0 ? "PASS" : "FAIL");
    printf("============================\n");

    cuMemFree(da);
    cuMemFree(db);
    cuMemFree(dc);

    cuModuleUnload(module);
    cuCtxDestroy(ctx);

    destroy(&program);

    dlclose(nvrtc);
    dlclose(cuda);

    free(ptx);
    free(a);
    free(b);
    free(c);

    return 0;
}

gpu_test();

CUDA devices: 16
Using: NVIDIA A100-SXM4-40GB MIG 1g.5gb

GPU COMPUTATION TEST
N          : 1024
GPU        : NVIDIA A100-SXM4-40GB MIG 1g.5gb
Verification: PASS


In [29]:
#include <stdio.h>
#include <stdlib.h>
#include <dlfcn.h>
#include <sys/time.h>

#define BENCH_N 10000000

double bench_time()
{
    struct timeval tv;
    gettimeofday(&tv, NULL);
    return (double)tv.tv_sec + (double)tv.tv_usec * 1.0e-6;
}

int gpu_benchmark2()
{
    /* =========================================================
       CUDA DRIVER API TYPES
       ========================================================= */

    typedef int CUresult;
    typedef int CUdevice;
    typedef void *CUcontext;
    typedef void *CUmodule;
    typedef void *CUfunction;
    typedef unsigned long long CUdeviceptr;

    #define CUDA_SUCCESS 0

    /* =========================================================
       LOAD CUDA DRIVER
       ========================================================= */

    void *cuda_lib = dlopen("libcuda.so.1", RTLD_NOW);

    if (!cuda_lib) {
        printf("CUDA driver failed: %s\n", dlerror());
        return 1;
    }

    CUresult (*p_cuInit)(unsigned int);
    CUresult (*p_cuDeviceGetCount)(int *);
    CUresult (*p_cuDeviceGet)(CUdevice *, int);
    CUresult (*p_cuDeviceGetName)(char *, int, CUdevice);
    CUresult (*p_cuCtxCreate)(CUcontext *, unsigned int, CUdevice);
    CUresult (*p_cuMemAlloc)(CUdeviceptr *, size_t);
    CUresult (*p_cuMemFree)(CUdeviceptr);
    CUresult (*p_cuMemcpyHtoD)(CUdeviceptr, const void *, size_t);
    CUresult (*p_cuMemcpyDtoH)(void *, CUdeviceptr, size_t);
    CUresult (*p_cuModuleLoadData)(CUmodule *, const void *);
    CUresult (*p_cuModuleGetFunction)(CUfunction *, CUmodule, const char *);
    CUresult (*p_cuLaunchKernel)(
        CUfunction,
        unsigned int, unsigned int, unsigned int,
        unsigned int, unsigned int, unsigned int,
        unsigned int,
        void *, void **, void *
    );
    CUresult (*p_cuCtxSynchronize)(void);
    CUresult (*p_cuModuleUnload)(CUmodule);
    CUresult (*p_cuCtxDestroy)(CUcontext);

    #define LOAD2(fn)                                      \
        p_##fn = dlsym(cuda_lib, #fn "_v2");               \
        if (!p_##fn) p_##fn = dlsym(cuda_lib, #fn);        \
        if (!p_##fn) {                                    \
            printf("Missing CUDA function: %s\n", #fn);   \
            return 1;                                     \
        }

    LOAD2(cuInit)
    LOAD2(cuDeviceGetCount)
    LOAD2(cuDeviceGet)
    LOAD2(cuDeviceGetName)
    LOAD2(cuCtxCreate)
    LOAD2(cuMemAlloc)
    LOAD2(cuMemFree)
    LOAD2(cuMemcpyHtoD)
    LOAD2(cuMemcpyDtoH)
    LOAD2(cuModuleLoadData)
    LOAD2(cuModuleGetFunction)
    LOAD2(cuLaunchKernel)
    LOAD2(cuCtxSynchronize)
    LOAD2(cuModuleUnload)
    LOAD2(cuCtxDestroy)

    p_cuInit(0);

    int device_count = 0;
    p_cuDeviceGetCount(&device_count);

    printf("CUDA devices visible: %d\n", device_count);

    CUdevice device;
    p_cuDeviceGet(&device, 0);

    char gpu_name[256];
    p_cuDeviceGetName(gpu_name, 256, device);

    printf("Using: %s\n", gpu_name);

    CUcontext context;

    if (p_cuCtxCreate(&context, 0, device) != CUDA_SUCCESS) {
        printf("Context creation failed\n");
        return 1;
    }

    /* =========================================================
       HOST DATA
       ========================================================= */

    int n = BENCH_N;

    float *A = malloc((size_t)n * sizeof(float));
    float *B = malloc((size_t)n * sizeof(float));
    float *C = malloc((size_t)n * sizeof(float));
    float *G = malloc((size_t)n * sizeof(float));

    if (!A || !B || !C || !G) {
        printf("Host allocation failed\n");
        return 1;
    }

    for (int i = 0; i < n; i++) {
        A[i] = (float)i;
        B[i] = 2.0f * (float)i;
    }

    /* =========================================================
       CPU TEST
       ========================================================= */

    double t0 = bench_time();

    for (int i = 0; i < n; i++) {
        C[i] = A[i] + B[i];
    }

    double cpu_time = bench_time() - t0;

    /* =========================================================
       LOAD NVRTC
       ========================================================= */

    void *nvrtc_lib = dlopen(
        "/srv/conda/envs/notebook/lib/python3.11/"
        "site-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12",
        RTLD_NOW
    );

    if (!nvrtc_lib) {
        printf("NVRTC failed: %s\n", dlerror());
        return 1;
    }

    typedef int (*nvrtcCreateProgram_t)(
        void **,
        const char *,
        const char *,
        int,
        const char **,
        const char **
    );

    typedef int (*nvrtcCompileProgram_t)(
        void *,
        int,
        const char **
    );

    typedef int (*nvrtcGetPTXSize_t)(
        void *,
        size_t *
    );

    typedef int (*nvrtcGetPTX_t)(
        void *,
        char *
    );

    typedef int (*nvrtcGetProgramLogSize_t)(
        void *,
        size_t *
    );

    typedef int (*nvrtcGetProgramLog_t)(
        void *,
        char *
    );

    typedef int (*nvrtcDestroyProgram_t)(
        void **
    );

    nvrtcCreateProgram_t p_create =
        dlsym(nvrtc_lib, "nvrtcCreateProgram");

    nvrtcCompileProgram_t p_compile =
        dlsym(nvrtc_lib, "nvrtcCompileProgram");

    nvrtcGetPTXSize_t p_get_ptx_size =
        dlsym(nvrtc_lib, "nvrtcGetPTXSize");

    nvrtcGetPTX_t p_get_ptx =
        dlsym(nvrtc_lib, "nvrtcGetPTX");

    nvrtcGetProgramLogSize_t p_get_log_size =
        dlsym(nvrtc_lib, "nvrtcGetProgramLogSize");

    nvrtcGetProgramLog_t p_get_log =
        dlsym(nvrtc_lib, "nvrtcGetProgramLog");

    nvrtcDestroyProgram_t p_destroy =
        dlsym(nvrtc_lib, "nvrtcDestroyProgram");

    if (!p_create || !p_compile || !p_get_ptx_size ||
        !p_get_ptx || !p_destroy) {

        printf("NVRTC functions missing\n");
        return 1;
    }

    /* =========================================================
       GPU KERNEL
       ========================================================= */

    const char *source =
        "extern \"C\" __global__ "
        "void add_arrays(float *a, float *b, float *c, int n) {"
        "    int i = blockIdx.x * blockDim.x + threadIdx.x;"
        "    if (i < n) c[i] = a[i] + b[i];"
        "}";

    void *program = NULL;

    if (p_create(
            &program,
            source,
            "add_arrays.cu",
            0,
            NULL,
            NULL) != 0) {

        printf("NVRTC program creation failed\n");
        return 1;
    }

    const char *options[] = {
        "--gpu-architecture=compute_80"
    };

    int compile_result =
        p_compile(program, 1, options);

    if (compile_result != 0) {

        printf("NVRTC compilation failed\n");

        if (p_get_log_size && p_get_log) {

            size_t log_size = 0;
            p_get_log_size(program, &log_size);

            char *log = malloc(log_size + 1);

            p_get_log(program, log);

            log[log_size] = '\0';

            printf("%s\n", log);

            free(log);
        }

        return 1;
    }

    size_t ptx_size = 0;

    p_get_ptx_size(program, &ptx_size);

    char *ptx = malloc(ptx_size);

    p_get_ptx(program, ptx);

    /* =========================================================
       LOAD GPU MODULE
       ========================================================= */

    CUmodule module;

    if (p_cuModuleLoadData(&module, ptx) != CUDA_SUCCESS) {
        printf("PTX module loading failed\n");
        return 1;
    }

    CUfunction kernel;

    p_cuModuleGetFunction(
        &kernel,
        module,
        "add_arrays"
    );

    /* =========================================================
       GPU MEMORY
       ========================================================= */

    CUdeviceptr dA;
    CUdeviceptr dB;
    CUdeviceptr dC;

    p_cuMemAlloc(
        &dA,
        (size_t)n * sizeof(float)
    );

    p_cuMemAlloc(
        &dB,
        (size_t)n * sizeof(float)
    );

    p_cuMemAlloc(
        &dC,
        (size_t)n * sizeof(float)
    );

    /* =========================================================
       HOST -> GPU
       ========================================================= */

    t0 = bench_time();

    p_cuMemcpyHtoD(
        dA,
        A,
        (size_t)n * sizeof(float)
    );

    p_cuMemcpyHtoD(
        dB,
        B,
        (size_t)n * sizeof(float)
    );

    p_cuCtxSynchronize();

    double h2d_time =
        bench_time() - t0;

    /* =========================================================
       GPU KERNEL
       ========================================================= */

    int threads = 256;
    int blocks =
        (n + threads - 1) / threads;

    void *kernel_args[] = {
        &dA,
        &dB,
        &dC,
        &n
    };

    t0 = bench_time();

    p_cuLaunchKernel(
        kernel,
        blocks, 1, 1,
        threads, 1, 1,
        0,
        NULL,
        kernel_args,
        NULL
    );

    p_cuCtxSynchronize();

    double gpu_kernel_time =
        bench_time() - t0;

    /* =========================================================
       GPU -> HOST
       ========================================================= */

    t0 = bench_time();

    p_cuMemcpyDtoH(
        G,
        dC,
        (size_t)n * sizeof(float)
    );

    p_cuCtxSynchronize();

    double d2h_time =
        bench_time() - t0;

    /* =========================================================
       VERIFICATION
       ========================================================= */

    int errors = 0;

    for (int i = 0; i < n; i++) {

        if (G[i] != C[i]) {

            errors++;

            if (errors <= 5) {
                printf(
                    "Mismatch %d: CPU=%f GPU=%f\n",
                    i,
                    C[i],
                    G[i]
                );
            }
        }
    }

    /* =========================================================
       RESULTS
       ========================================================= */

    printf("\n");
    printf("========================================\n");
    printf("GPU BENCHMARK\n");
    printf("========================================\n");

    printf("N               : %d\n", n);
    printf("GPU             : %s\n", gpu_name);

    printf("\n");

    printf(
        "CPU computation : %.6f s\n",
        cpu_time
    );

    printf(
        "GPU kernel      : %.6f s\n",
        gpu_kernel_time
    );

    printf(
        "CPU -> GPU      : %.6f s\n",
        h2d_time
    );

    printf(
        "GPU -> CPU      : %.6f s\n",
        d2h_time
    );

    double gpu_total =
        h2d_time +
        gpu_kernel_time +
        d2h_time;

    printf(
        "GPU total       : %.6f s\n",
        gpu_total
    );

    printf(
        "Kernel speedup  : %.2fx\n",
        cpu_time / gpu_kernel_time
    );

    printf(
        "Total speedup   : %.2fx\n",
        cpu_time / gpu_total
    );

    printf(
        "Verification    : %s\n",
        errors == 0 ? "PASS" : "FAIL"
    );

    printf("========================================\n");

    /* =========================================================
       CLEANUP
       ========================================================= */

    p_cuMemFree(dA);
    p_cuMemFree(dB);
    p_cuMemFree(dC);

    p_cuModuleUnload(module);
    p_cuCtxDestroy(context);

    p_destroy(&program);

    dlclose(nvrtc_lib);
    dlclose(cuda_lib);

    free(ptx);

    free(A);
    free(B);
    free(C);
    free(G);

    return 0;
}

gpu_benchmark2();

CUDA devices visible: 16
Using: NVIDIA A100-SXM4-40GB MIG 1g.5gb

GPU BENCHMARK
N               : 10000000
GPU             : NVIDIA A100-SXM4-40GB MIG 1g.5gb

CPU computation : 0.026326 s
GPU kernel      : 0.000025 s
CPU -> GPU      : 0.007740 s
GPU -> CPU      : 0.008064 s
GPU total       : 0.015829 s
Kernel speedup  : 1051.61x
Total speedup   : 1.66x
Verification    : PASS


ERROR: received bad message: No such comm registered: 71dcf802-6fad-4220-9a69-7ae73c50603a
Message type: comm_msg
